# 07 --  Model Explainability

## Concept
Models must be explainable. Stakeholders need to trust predictions and understand why a decision was made.

## Mathematical Intuition
- **SHAP**: Shapley values from cooperative game theory. Each feature gets a payout = its contribution to the prediction, averaged over all possible feature orderings.
- **Permutation Importance**: Shuffle a feature -> measure accuracy drop -> that's the feature's importance.
- **Feature Importance**: Tree-based models track how much each feature reduces impurity (Gini or entropy).

## Interview Questions
1. Why are SHAP values considered theoretically fair?
2. What's the difference between global and local interpretability?
3. When would permutation importance give misleading results?

## Production Mapping
Explainers are in `pipeline/explainability.py` --  FeatureImportanceExplainer, PermutationImportanceExplainer, ShapExplainer.


In [ ]:
import pandas as pd, numpy as np, matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import RandomForestClassifier
np.random.seed(42)
n = 1000
X = pd.DataFrame({
    'capacity_mw': np.random.exponential(500, n),
    'region_risk': np.random.uniform(0, 1, n),
    'age_years': np.random.exponential(30, n),
    'num_connections': np.random.poisson(5, n),
})
y = (X['capacity_mw'] / 100 + X['region_risk'] * 5 + np.random.normal(0, 0.5, n)).clip(0, 3).round().astype(int).clip(0, 3)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
scaler = StandardScaler()
X_train_s = scaler.fit_transform(X_train)
X_test_s = scaler.transform(X_test)
model = RandomForestClassifier(n_estimators=100, random_state=42)
model.fit(X_train_s, y_train)
print(f"Test accuracy: {model.score(X_test_s, y_test):.4f}")

In [ ]:
# Built-in feature importance
importances = model.feature_importances_
indices = np.argsort(importances)[::-1]
plt.figure(figsize=(8, 4))
plt.bar(range(len(importances)), importances[indices])
plt.xticks(range(len(importances)), [X.columns[i] for i in indices], rotation=45)
plt.title('Random Forest Feature Importance')
plt.tight_layout()
plt.savefig('../artifacts/07_feature_importance.png', dpi=100)
plt.show()

In [ ]:
# Permutation Importance
from sklearn.inspection import permutation_importance
perm = permutation_importance(model, X_test_s, y_test, n_repeats=10, random_state=42)
perm_df = pd.DataFrame({
    'feature': X.columns,
    'importance_mean': perm.importances_mean,
    'importance_std': perm.importances_std
}).sort_values('importance_mean', ascending=False)
print(perm_df.to_string(index=False))

In [ ]:
# SHAP
import shap
explainer = shap.TreeExplainer(model)
shap_values = explainer.shap_values(X_test_s[:100])

# Summary plot
plt.figure()
shap.summary_plot(shap_values, X_test_s[:100], feature_names=X.columns, show=False)
plt.tight_layout()
plt.savefig('../artifacts/07_shap_summary.png', dpi=100, bbox_inches='tight')
plt.show()

In [ ]:
# Force plot for a single prediction (binary for simplicity)
# For multiclass, show the force for class with highest prob
# We'll use a simplified 2-class version for demonstration
y_bin = (y > 1).astype(int)
Xb_train, Xb_test, yb_train, yb_test = train_test_split(X, y_bin, test_size=0.2, random_state=42)
scaler_b = StandardScaler()
Xb_train_s = scaler_b.fit_transform(Xb_train)
Xb_test_s = scaler_b.transform(Xb_test)
model_bin = RandomForestClassifier(n_estimators=100, random_state=42)
model_bin.fit(Xb_train_s, yb_train)

explainer_bin = shap.TreeExplainer(model_bin)
shap_bin = explainer_bin.shap_values(Xb_test_s[:1])
shap.force_plot(explainer_bin.expected_value, shap_bin[0], Xb_test[:1], feature_names=X.columns, matplotlib=True)
plt.savefig('../artifacts/07_shap_force.png', dpi=100, bbox_inches='tight')
plt.show()

## Key Takeaways
- SHAP provides both global and local explanations
- Permutation importance is model-agnostic
- Feature importances from tree models are fast but biased toward high-cardinality features
- Always explain predictions for production deployments
- Production inference returns prediction + confidence + probabilities + model metadata